## 1) Clean transcript 

In [8]:
import re
transcript_file = "stream_transcripts/full_transcript.txt"

with open(transcript_file, "r", encoding="utf-8") as f:
    all_text = f.read()

# basic cleanup for web junk / repeated nav phrases
all_text = re.sub(r"http\\S+", "", all_text)
all_text = re.sub(r"(back to .*?\\.)+", "", all_text, flags=re.IGNORECASE)

# Split by sentence
sentences = re.split(r'(?<=[.!?]) +', all_text)

# Chunk ~120 words per chunk
chunks = []
current = ""
for s in sentences:
    if len(current.split()) + len(s.split()) > 120:
        chunks.append(current.strip())
        current = ""
    current += s + " "
if current:
    chunks.append(current.strip())

print(f"{len(chunks)} chunks created for RAG.")


4 chunks created for RAG.


In [13]:
import re

with open(transcript_file, "r", encoding="utf-8") as f:
    text = f.read()

sentences = re.split(r'(?<=[.!?]) +', text)

# Simple keyword rules (tune as needed)
subjective_kw = ["i have", "i've", "i feel", "my", "cough", "chest", "tired", "run down", "phlegm", "fever", "chills"]
objective_kw  = ["lungs", "listen", "stethoscope", "breath", "exam", "tapping", "vital"]
assessment_kw = ["diagnosis", "assessment", "bronchitis", "infection", "asthma", "allerg"]
plan_kw       = ["prescribe", "antibiotic", "medicine", "rest", "fluids", "follow-up", "reassess", "emergency", "go to"]

def pick_sentences(kws):
    out = []
    for s in sentences:
        s_low = s.lower()
        if any(k in s_low for k in kws):
            out.append(s.strip())
    return out

S = pick_sentences(subjective_kw)
O = pick_sentences(objective_kw)
A = pick_sentences(assessment_kw)
P = pick_sentences(plan_kw)

soap_notes = "\n".join([
    "S: " + (" ".join(S) if S else "Not mentioned."),
    "O: " + (" ".join(O) if O else "Not mentioned."),
    "A: " + (" ".join(A) if A else "Not mentioned."),
    "P: " + (" ".join(P) if P else "Not mentioned."),
])

print(soap_notes)


S: It's this cough. I've had it for about three weeks now, and it just won't go away. Can you describe the cough for me? It's mostly dry, but sometimes, especially in the mornings, I cough up a little bit of phlegm. Any fever, chills, or body aches? No fever, but I felt a bit run down, a little tired, and my chest feels tight, like a weight on my chest. Sound of tapping on chest. I've generally been pretty healthy. I'm going to prescribe you some cough medicine to help loosen the mucus, and I'll also write you a prescription for an antibiotic just in case it's bacterial. I've heard that doctors prescribe them too much. If you develop a high fever, over 102 degrees Fahrenheit, or if you have trouble breathing, please go to the emergency room immediately.
O: All right, let's have a listen to your lungs. Sound of stethoscope? Take a deep breath in, and out, good, again, and out, okay. Sound of tapping on chest. If you develop a high fever, over 102 degrees Fahrenheit, or if you have troub

In [14]:
print("Num chunks:", len(chunks))
print("Sample chunk:", chunks[0][:300])

test_context = retrieve("What symptoms does the patient report?", k=6)
print("Retrieved context length:", len(test_context))
print("Retrieved sample:", test_context[0][:300] if test_context else "EMPTY")


Num chunks: 4
Sample chunk: Doctor, thanks for seeing me so quickly. Good morning, Zahra. Please, have a seat. So what brings you in today? It's this cough. I've had it for about three weeks now, and it just won't go away. It's really starting to bother me. Okay, three weeks is definitely a good amount of time to see if it wil
Retrieved context length: 4
Retrieved sample: Doctor, thanks for seeing me so quickly. Good morning, Zahra. Please, have a seat. So what brings you in today? It's this cough. I've had it for about three weeks now, and it just won't go away. It's really starting to bother me. Okay, three weeks is definitely a good amount of time to see if it wil


In [ ]:
import re

with open(transcript_file, "r", encoding="utf-8") as f:
    all_text = f.read()

# basic cleanup for web junk / repeated nav phrases
all_text = re.sub(r"http\S+", "", all_text)
all_text = re.sub(r"(back to .*?\.)+", "", all_text, flags=re.IGNORECASE)

# Split by sentence
sentences = re.split(r'(?<=[.!?]) +', all_text)

# Chunk ~120 words per chunk
chunks = []
current = ""
for s in sentences:
    if len(current.split()) + len(s.split()) > 120:
        chunks.append(current.strip())
        current = ""
    current += s + " "
if current:
    chunks.append(current.strip())

print(f"{len(chunks)} chunks created for RAG.")

In [ ]:
from pathlib import Path

kb_dir = Path("kb")
kb_dir.mkdir(exist_ok=True)

kb_files = {
    "soap_template.txt": """
SOAP Notes Template
S (Subjective): Patient-reported symptoms, duration, concerns, relevant history.
O (Objective): Exam findings, observations, vitals, tests if mentioned.
A (Assessment): Clinician assessment/diagnosis.
P (Plan): Medications, advice, follow-up, red flags.
""".strip(),
    "cough_bronchitis.txt": """
Cough lasting >2-3 weeks can suggest bronchitis.
Dry or productive cough with yellow phlegm can be consistent with bronchitis.
Chest tightness and fatigue are common in bronchitis.
If bacterial infection is suspected, antibiotics may be considered.
""".strip(),
    "red_flags.txt": """
Red flags for respiratory illness: high fever (>102F), shortness of breath, chest pain.
If red flags are present, patient should seek urgent or emergency care.
""".strip(),
    "care_advice.txt": """
General advice: rest, fluids, avoid irritants like smoke or strong perfumes.
Follow-up if symptoms do not improve in a few days.
""".strip(),
}

for name, text in kb_files.items():
    (kb_dir / name).write_text(text, encoding="utf-8")

print(f"KB files created in: {kb_dir}")

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss
from pathlib import Path

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Load KB docs
kb_paths = list(Path("kb").glob("*.txt"))
kb_texts = [p.read_text(encoding="utf-8") for p in kb_paths]

# Build combined corpus
all_docs = chunks + kb_texts

doc_embeddings = embedder.encode(all_docs, normalize_embeddings=True)
doc_embeddings = np.array(doc_embeddings, dtype="float32")

index = faiss.IndexFlatIP(doc_embeddings.shape[1])
index.add(doc_embeddings)

# Track source for debugging (transcript vs KB)
all_sources = ["transcript"] * len(chunks) + [p.name for p in kb_paths]

def retrieve(query, k=8):
    q_emb = embedder.encode([query], normalize_embeddings=True).astype("float32")
    scores, idx = index.search(q_emb, k)
    return [(all_docs[i], all_sources[i]) for i in idx[0] if i != -1]

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

device = "mps" if torch.backends.mps.is_available() else "cpu"

# Stronger model for instruction following
model_name = "google/flan-t5-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)


def generate_soap_section(section_name, question, k=8):
    hits = retrieve(question, k=k)
    context = "
".join([h[0] for h in hits])

    prompt = f"""
You are a medical scribe. Use ONLY the context below.
If information is not present, write: Not mentioned.

Context:
{context}

Write the {section_name} section of SOAP notes as a concise paragraph.
Do NOT repeat phrases. Do NOT add anything not in the context.
"""

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(device)
    outputs = model.generate(
        **inputs,
        max_length=220,
        min_length=40,
        do_sample=False,
        no_repeat_ngram_size=3,
        repetition_penalty=1.3,
        num_beams=4
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

soap = []
soap.append(generate_soap_section("S", "What symptoms, concerns, duration, and history are reported?"))
soap.append(generate_soap_section("O", "What exam findings or objective observations are mentioned?"))
soap.append(generate_soap_section("A", "What assessment or diagnosis is given?"))
soap.append(generate_soap_section("P", "What treatments, medications, advice, and follow-up are mentioned?"))

soap_notes = "
".join(soap)
print(soap_notes)